# Desarrollo MLP: predicción y clustering en validation

Este notebook evalúa el MLP primario `32→64→64→32` sobre datos sintéticos nuevos. Selecciona checkpoints sólo mediante predicción y no-colapso; luego mide clustering en validation. **No construye ni consulta test.**

## Protocolo congelado

La configuración usa `base_seed=1`, 64/32 masters train/validation por régimen, seeds 10–14 y 20 épocas. La pureza de cada modelo es la media de 20 `random_state` de K-means con `n_init=20`. El gate exige media global ≥60%, peor seed ≥55%, CV entre seeds ≤10% y sd intra-seed ≤3 puntos. `65.48%` se muestra sólo como referencia descriptiva del paper.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import (
    PaperCheckpointReplayConfig,
    load_paper_mlp_clustering_development_config,
)
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_evaluation import (
    evaluate_paper_mlp_clustering_gate,
    evaluate_paper_mlp_seed_clustering,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    evaluate_scale_invariant_seed_stability_gate,
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    summarize_seed_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_mlp_clustering_development.yaml"
config = load_paper_mlp_clustering_development_config(config_path)
replay_config = PaperCheckpointReplayConfig(metric_absolute_tolerance=1e-8)
torch.use_deterministic_algorithms(True)
print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {validation_dataset.sample_key(index) for index in range(len(validation_dataset))}
assert config.data.base_seed == 1
assert len(train_dataset) == 64 * len(PAPER_REGIME_NAMES) == 1152
assert len(validation_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation nuevos: {len(train_dataset)}/{len(validation_dataset)}")
print("Test no fue instanciado.")

In [ ]:
models, replay_results, summaries, histories, times = {}, {}, [], {}, {}
checkpoint_diagnostics = {}
for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)
    started = time.perf_counter()
    checkpoint_run = run_paper_train_validation_with_checkpoint(
        model, train_dataset, validation_dataset, run_config, config.checkpoint_gate
    )
    histories[seed] = checkpoint_run.history
    initial = checkpoint_run.history[0]
    diagnostic_rows = []
    for row in checkpoint_run.history[1:]:
        loss_ratio = row.validation_loss / max(initial.validation_loss, 1e-12)
        gap = row.validation_loss / max(row.train_loss, 1e-12)
        spread_ratio = row.validation_embedding_std / max(initial.validation_embedding_std, 1e-12)
        checks = (
            loss_ratio <= config.checkpoint_gate.max_validation_loss_ratio,
            gap <= config.checkpoint_gate.max_validation_train_loss_ratio,
            spread_ratio >= config.checkpoint_gate.min_validation_embedding_std_ratio,
            row.validation_effective_rank >= config.checkpoint_gate.min_validation_effective_rank,
        )
        diagnostic_rows.append({
            "epoch": row.epoch, "validation_loss": row.validation_loss,
            "loss_ratio": loss_ratio, "gap": gap, "spread_ratio": spread_ratio,
            "rank": row.validation_effective_rank, "checks": checks,
            "passed_count": sum(checks),
        })
    checkpoint_diagnostics[seed] = min(
        diagnostic_rows, key=lambda item: (-item["passed_count"], item["validation_loss"], item["epoch"])
    )
    if checkpoint_run.selection is None:
        times[seed] = time.perf_counter() - started
        closest = checkpoint_diagnostics[seed]
        print(
            f"seed={seed} checkpoint=NONE closest_epoch={closest['epoch']} "
            f"checks={closest['passed_count']}/4 val/base={closest['loss_ratio']:.3f} "
            f"gap={closest['gap']:.3f} spread={closest['spread_ratio']:.3f} "
            f"rank={closest['rank']:.2f} time={times[seed]:.2f}s"
        )
        continue
    replay = verify_paper_checkpoint_replay(
        model, checkpoint_run, validation_dataset, run_config, replay_config,
        expected_epoch=checkpoint_run.selection.epoch,
    )
    times[seed] = time.perf_counter() - started
    models[seed], replay_results[seed] = model, replay
    summaries.append(summarize_seed_checkpoint(seed, checkpoint_run.selection))
    print(
        f"seed={seed} checkpoint={checkpoint_run.selection.epoch} "
        f"val/base={checkpoint_run.selection.validation_loss_ratio:.3f} "
        f"gap={checkpoint_run.selection.validation_train_loss_ratio:.3f} "
        f"rank={checkpoint_run.selection.validation_effective_rank:.2f} "
        f"replay={'PASS' if replay.passed else 'FAIL'} time={times[seed]:.2f}s"
    )
replay_passed = set(replay_results) == set(config.sweep.seeds) and all(row.passed for row in replay_results.values())
stability_gate = evaluate_scale_invariant_seed_stability_gate(summaries, config.sweep, config.stability_gate)
predictive_prerequisites_passed = replay_passed and stability_gate.passed
print(f"Replay global: {'PASS' if replay_passed else 'FAIL'}; tiempo total={sum(times.values()):.2f}s")
print("Gate predictivo:", json.dumps(asdict(stability_gate), indent=2))
print("Test sigue sin instanciar.")

In [ ]:
@torch.no_grad()
def collect_validation_embeddings(model, run_config):
    device = torch.device(run_config.device)
    model.to(device).eval()
    loader = make_paper_loader(validation_dataset, run_config, shuffle=False)
    embedding_chunks, label_chunks = [], []
    for context, _, labels in loader:
        embedding_chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
        label_chunks.append(labels.numpy())
    return np.concatenate(embedding_chunks), np.concatenate(label_chunks)

clustering_metrics, validation_embeddings_by_seed = [], {}
clustering_gate = None
if predictive_prerequisites_passed:
    validation_labels = None
    for seed in config.sweep.seeds:
        embeddings, labels = collect_validation_embeddings(models[seed], replace(config.train, seed=seed))
        validation_labels = labels if validation_labels is None else validation_labels
        assert np.array_equal(labels, validation_labels)
        validation_embeddings_by_seed[seed] = embeddings
        clustering_metrics.append(evaluate_paper_mlp_seed_clustering(embeddings, labels, seed, config.clustering))
    clustering_gate = evaluate_paper_mlp_clustering_gate(clustering_metrics, config.sweep, config.clustering_gate)
    print("Gate de clustering:", json.dumps(asdict(clustering_gate), indent=2))
else:
    print("Clustering OMITIDO: no pasaron todos los prerrequisitos predictivos.")
development_passed = predictive_prerequisites_passed and clustering_gate is not None and clustering_gate.passed
print(f"Gate conjunto: {'PASS' if development_passed else 'FAIL'}")
print("Test no fue instanciado ni consultado.")

In [ ]:
summary_by_seed = {row.seed: row for row in summaries}
clustering_by_seed = {row.seed: row for row in clustering_metrics}
print("seed selected closest checks val/base gap spread rank purity_mean purity_sd")
for seed in config.sweep.seeds:
    diagnostic = checkpoint_diagnostics[seed]
    selected = summary_by_seed.get(seed)
    cluster = clustering_by_seed.get(seed)
    selected_epoch = str(selected.checkpoint_epoch) if selected is not None else "NONE"
    purity = f"{cluster.mean_purity:.2%}" if cluster is not None else "not-run"
    purity_sd = f"{cluster.purity_std:.2%}" if cluster is not None else "not-run"
    print(
        f"{seed:>4d} {selected_epoch:>8s} {diagnostic['epoch']:>7d} {diagnostic['passed_count']}/4 "
        f"{diagnostic['loss_ratio']:>8.3f} {diagnostic['gap']:>5.3f} "
        f"{diagnostic['spread_ratio']:>6.3f} {diagnostic['rank']:>5.2f} "
        f"{purity:>11s} {purity_sd:>9s}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
colors = plt.cm.tab10(np.linspace(0.0, 1.0, len(config.sweep.seeds)))
for color, seed in zip(colors, config.sweep.seeds, strict=True):
    history = histories[seed]
    epochs = np.array([row.epoch for row in history])
    validation_loss = np.array([row.validation_loss for row in history])
    train_loss = np.array([row.train_loss for row in history])
    ranks = np.array([row.validation_effective_rank for row in history])
    axes[0, 0].plot(epochs, validation_loss / validation_loss[0], marker="o", markersize=3, color=color, label=f"seed {seed}")
    axes[0, 1].plot(epochs, validation_loss / np.maximum(train_loss, 1e-12), marker="o", markersize=3, color=color)
    axes[0, 2].plot(epochs, ranks, marker="o", markersize=3, color=color)
axes[0, 0].axhline(config.checkpoint_gate.max_validation_loss_ratio, color="tab:red", linestyle="--")
axes[0, 0].set(title="Validation loss / baseline", xlabel="Época", ylabel="Ratio")
axes[0, 0].legend(ncol=2, fontsize=8)
axes[0, 1].axhline(config.checkpoint_gate.max_validation_train_loss_ratio, color="tab:red", linestyle="--")
axes[0, 1].set(title="Brecha validation/train", xlabel="Época", ylabel="Ratio")
axes[0, 2].axhline(config.checkpoint_gate.min_validation_effective_rank, color="tab:red", linestyle="--")
axes[0, 2].set(title="Rango efectivo validation", xlabel="Época", ylabel="Rango")
seeds = np.array(config.sweep.seeds)
if clustering_gate is not None:
    means = np.array([clustering_by_seed[seed].mean_purity for seed in seeds])
    stds = np.array([clustering_by_seed[seed].purity_std for seed in seeds])
    matched = np.array([clustering_by_seed[seed].mean_matched_accuracy for seed in seeds])
    axes[1, 0].errorbar(seeds, means, yerr=stds, fmt="o", capsize=5)
    axes[1, 0].axhline(config.clustering_gate.min_worst_seed_mean_purity, color="tab:red", linestyle="--", label="mínimo por seed")
    axes[1, 0].axhline(0.6548, color="tab:green", linestyle=":", label="paper (descriptivo)")
    axes[1, 0].set(title="Pureza media ± sd K-means", xlabel="Seed", ylabel="Pureza")
    axes[1, 0].legend(fontsize=8)
    axes[1, 1].boxplot([clustering_by_seed[seed].purities for seed in seeds], tick_labels=seeds)
    axes[1, 1].axhline(config.clustering_gate.min_overall_mean_purity, color="tab:red", linestyle="--")
    axes[1, 1].set(title="Distribución por K-means state", xlabel="Seed modelo", ylabel="Pureza")
    width = 0.36
    axes[1, 2].bar(seeds - width / 2, means, width=width, label="pureza")
    axes[1, 2].bar(seeds + width / 2, matched, width=width, label="matched")
    axes[1, 2].set(title="Métricas de clustering", xlabel="Seed", ylabel="Score")
    axes[1, 2].legend()
else:
    for color, seed in zip(colors, config.sweep.seeds, strict=True):
        history = histories[seed]
        initial = history[0]
        epochs = np.array([row.epoch for row in history[1:]])
        spread = np.array([row.validation_embedding_std / max(initial.validation_embedding_std, 1e-12) for row in history[1:]])
        passed_counts = np.array([
            sum((
                row.validation_loss / max(initial.validation_loss, 1e-12) <= config.checkpoint_gate.max_validation_loss_ratio,
                row.validation_loss / max(row.train_loss, 1e-12) <= config.checkpoint_gate.max_validation_train_loss_ratio,
                row.validation_embedding_std / max(initial.validation_embedding_std, 1e-12) >= config.checkpoint_gate.min_validation_embedding_std_ratio,
                row.validation_effective_rank >= config.checkpoint_gate.min_validation_effective_rank,
            ))
            for row in history[1:]
        ])
        axes[1, 0].plot(epochs, spread, marker="o", markersize=3, color=color, label=f"seed {seed}")
        axes[1, 1].plot(epochs, passed_counts, marker="o", markersize=3, color=color)
    axes[1, 0].axhline(config.checkpoint_gate.min_validation_embedding_std_ratio, color="tab:red", linestyle="--")
    axes[1, 0].set(title="Embedding std / baseline", xlabel="Época", ylabel="Ratio")
    axes[1, 0].legend(ncol=2, fontsize=8)
    axes[1, 1].axhline(4, color="tab:red", linestyle="--")
    axes[1, 1].set(title="Restricciones predictivas cumplidas", xlabel="Época", ylabel="Cantidad (0–4)", ylim=(0, 4.2))
    axes[1, 2].axis("off")
    missing_lines = [
        f"seed {seed}: época {checkpoint_diagnostics[seed]['epoch']}, {checkpoint_diagnostics[seed]['passed_count']}/4"
        for seed in config.sweep.seeds if seed not in models
    ]
    axes[1, 2].text(0.02, 0.95, "Clustering omitido\n\n" + "\n".join(missing_lines), va="top", fontsize=12)
plt.show()

In [ ]:
status = "PASS" if development_passed else "FAIL"
criteria = {"replay completo": replay_passed, "estabilidad predictiva": stability_gate.passed}
if clustering_gate is not None:
    criteria.update({
        "media global": clustering_gate.overall_mean_passed,
        "peor seed": clustering_gate.worst_seed_passed,
        "variabilidad entre modelos": clustering_gate.seed_variability_passed,
        "estabilidad K-means": clustering_gate.kmeans_stability_passed,
    })
failed_text = ", ".join(name for name, passed in criteria.items() if not passed) or "ninguno"
failed_seeds = clustering_gate.failed_seeds if clustering_gate is not None else tuple(seed for seed in config.sweep.seeds if seed not in models)
failed_seed_text = ", ".join(str(seed) for seed in failed_seeds) or "ninguna"
selected_epochs = ", ".join(f"{row.seed}:{row.checkpoint_epoch}" for row in summaries)
if clustering_gate is None:
    clustering_text = "- **Clustering:** no medido; se omitió porque faltaron checkpoints predictivos elegibles."
    decision = "El protocolo queda bloqueado antes de clustering. Hay que analizar las curvas de train/validation y definir una nueva corrida de desarrollo explícita, sin abrir test."
else:
    clustering_text = (
        f"- **Pureza media global:** `{clustering_gate.overall_mean_purity:.2%}` / mínimo `{config.clustering_gate.min_overall_mean_purity:.0%}`.\n"
        f"- **Peor seed:** `{clustering_gate.worst_seed_mean_purity:.2%}` / mínimo `{config.clustering_gate.min_worst_seed_mean_purity:.0%}`.\n"
        f"- **CV entre seeds:** `{clustering_gate.seed_mean_purity_coefficient_of_variation:.3f}` / máximo `{config.clustering_gate.max_seed_mean_purity_coefficient_of_variation:.2f}`.\n"
        f"- **Peor sd K-means:** `{clustering_gate.worst_within_seed_purity_std:.2%}` / máximo `{config.clustering_gate.max_within_seed_purity_std:.0%}`."
    )
    decision = (
        "El desarrollo MLP pasa. Podemos congelar estos epochs y diseñar una evaluación held-out nueva."
        if development_passed
        else "El desarrollo MLP falla. Debemos diagnosticar train/validation sin abrir test ni retocar thresholds."
    )
display(Markdown(f"""## Resultado e interpretación

- **Gate conjunto: {status}.**
- **Criterios fallidos:** {failed_text}.
- **Seeds fallidas:** {failed_seed_text}.
- **Epochs seleccionados:** `{selected_epochs}`.
{clustering_text}

Los paneles superiores verifican predicción y rango. Los inferiores muestran clustering si se habilitó; de lo contrario, localizan el rechazo predictivo. `65.48%` es sólo referencia descriptiva porque escala y split no coinciden.

### Decisión

{decision} Test no fue construido ni consultado.
"""))